# 9. ResNet ERMにおける陽性率の異なる群の性能差

DROや公平性手法を比べる前提として、**ERMの時点で、陽性率の違う群のあいだに何が起きているか**を確かめる。

陽性率の違いがTPR・TNRに出る経路は、モデルが群の陽性率をスコアに反映しているかどうかで2通りに分かれる。

- **反映していない（切片を全群で共有）**：陽性率の低い群ではスコアが高すぎ、高い群では低すぎる。較正が群ごとにずれる。
- **反映している（群ごとに切片を変えている）**：較正は群ごとに合う。ただし陽性率の低い群ではスコア全体が下がるので、全群共通の閾値ではTPRが下がりTNRが上がる。

どちらでもTPRとTNRは逆向きに開くので、TPR・TNRだけでは区別できない。本notebookでは、モデルが見積もる群の陽性率を実際の陽性率と比べて区別する。

対象はResNet ERM（global inverse class weight、条件ID M01）の3seed（42・43・44）。best validation AUROC checkpointを、race-knownのtestで評価する。

群は2つの属性を**別々に**切る。交差群は扱わない。

- 年齢：<65 / ≥65（DRO群と同じ境界）
- race：White / Asian / Black（Othersは除く）

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import rootutils

ROOT = rootutils.setup_root(Path.cwd(), indicator=".project-root", pythonpath=True)
from analysis.common.paths import STYLE_SHEET, study_dir  # noqa: E402
from analysis.common.run_artifacts import read_config  # noqa: E402
from analysis.archive.race_known_architecture_objective_comparison.comparison import (  # noqa: E402
    performance,
    prediction_command,
    read_predictions,
    select_runs,
)

plt.style.use(STYLE_SHEET)
PACKAGE = study_dir("race_known_architecture_objective_comparison")
TOPIC = "09_erm_prevalence_by_subgroup"
SEEDS = [42, 43, 44]
CONDITIONS = {"M01": "resnet-chexpert-race-known"}
MODEL_NAME = "ResNet ERM / global inverse class weight"
SUBGROUPS = [("age", "<65"), ("age", "≥65"), ("race", "White"), ("race", "Asian"), ("race", "Black")]
ATTRIBUTE_COLORS = {"age": "C0", "race": "C1"}
RESULTS = PACKAGE / "results" / TOPIC
FIGURES = PACKAGE / "figures" / TOPIC
RESULTS.mkdir(parents=True, exist_ok=True)
FIGURES.mkdir(parents=True, exist_ok=True)

## 入力

成功runとbest checkpoint、train/valのSHA256を学習記録と照合する（`select_runs`）。cacheが欠けていれば、表示されたコマンドをrepo rootで実行する（GPUは自動では使わない）。

class weightは3seedで同じ値であることを確かめる。後半で、このweightが作るスコアの一律のずれを差し引くのに使う。

In [ ]:
provenance, frame, directories = select_runs(PACKAGE, CONDITIONS, SEEDS)
provenance.to_csv(RESULTS / "provenance.csv", index=False)
command = prediction_command(PACKAGE, directories, device="cuda:0")
print(command or "全予測cacheが有効")
caches = read_predictions(PACKAGE, directories, frame)

configs = [read_config(path / "config.yaml") for path in directories.values()]
class_weights = {tuple(config["model"]["loss_fn"]["class_weight"]) for config in configs}
if len(class_weights) != 1:
    raise ValueError(f"seed間でclass weightが異なる: {class_weights}")
CLASS_WEIGHT = np.array(class_weights.pop())
WEIGHT_LOG_ODDS = float(np.log(CLASS_WEIGHT[1] / CLASS_WEIGHT[0]))
train = pd.read_csv(Path(provenance.iloc[0]["split_path"]).with_name("train.csv"))
print(f"class weight: {CLASS_WEIGHT}, log(w1/w0) = {WEIGHT_LOG_ODDS:.3f}")
print(f"test: {len(frame)}件、train: {len(train)}件")

## 群の陽性率

まず、群ごとの陽性率がどれだけ違うかを見る。モデルが学習で見るのはtrainの陽性率なので、trainとtestを並べる。年齢とraceのどちらで差が大きいかが、以降の読み方の基準になる。

In [ ]:
def subgroup_masks(split):
    """年齢2区分とrace3区分の所属を、属性ごとに独立に返す。

    交差群にしないのは、各属性の陽性率の差が性能差にどう出るかを、セルの小ささに邪魔されずに見るため。
    race_groupの0・1・2がWhite・Asian・Blackで、3（Others）は混成なので除く。

    Args:
        split: race-knownのsplit。
    Returns:
        dict: (属性, 群名)からbool配列への対応。
    """
    age = split["age"].to_numpy()
    age_known = ~split["age_missing"].astype(bool).to_numpy() & np.isfinite(age)
    race = split["race_group"].to_numpy()
    race_known = ~split["race_group_missing"].astype(bool).to_numpy()
    return {
        ("age", "<65"): age_known & (age < 65),
        ("age", "≥65"): age_known & (age >= 65),
        ("race", "White"): race_known & (race == 0),
        ("race", "Asian"): race_known & (race == 1),
        ("race", "Black"): race_known & (race == 2),
    }


test_masks = subgroup_masks(frame)
train_masks = subgroup_masks(train)
prevalence_rows = []
for split_name, split, masks in [("train", train, train_masks), ("test", frame, test_masks)]:
    target = split["target"].to_numpy()
    prevalence_rows.append(
        {
            "split": split_name,
            "attribute": "overall",
            "subgroup": "overall",
            "n": len(target),
            "n_positive": int(target.sum()),
            "prevalence": target.mean(),
        }
    )
    for (attribute, subgroup), mask in masks.items():
        prevalence_rows.append(
            {
                "split": split_name,
                "attribute": attribute,
                "subgroup": subgroup,
                "n": int(mask.sum()),
                "n_positive": int(target[mask].sum()),
                "prevalence": target[mask].mean(),
            }
        )
prevalence = pd.DataFrame(prevalence_rows)
prevalence.to_csv(RESULTS / "prevalence.csv", index=False)

fig, ax = plt.subplots(figsize=(7, 3.5))
positions = np.arange(len(SUBGROUPS))
for offset, split_name, hatch in [(-0.2, "train", ""), (0.2, "test", "//")]:
    rows = prevalence[prevalence.split == split_name].set_index(["attribute", "subgroup"]).loc[SUBGROUPS]
    colors = [ATTRIBUTE_COLORS[attribute] for attribute, _ in SUBGROUPS]
    ax.bar(
        positions + offset,
        rows["prevalence"],
        width=0.4,
        color=colors,
        hatch=hatch,
        edgecolor="white",
        label=split_name,
    )
for split_name, style in [("train", "-"), ("test", "--")]:
    overall = prevalence.query("split == @split_name and attribute == 'overall'")["prevalence"].item()
    ax.axhline(overall, color="gray", linestyle=style, linewidth=1, label=f"overall ({split_name})")
ax.set_xticks(positions, [f"{attribute}\n{subgroup}" for attribute, subgroup in SUBGROUPS])
ax.set_ylabel("positive rate")
ax.set_title("Positive rate by subgroup (solid: train, hatched: test)")
ax.legend(fontsize=8, ncol=2)
fig.tight_layout()
fig.savefig(FIGURES / "prevalence.png", dpi=160)
plt.show()

年齢では陽性率が<65で14.4%、≥65で5.7%と2.5倍違う（test、trainもほぼ同じ）。raceは9.5〜12.2%で、差は小さい。

この差が性能にどう出るかを次に見る。AUROCは群内の順位だけで決まり、TPR・TNRは全群共通の閾値（argmax、class weight込み）での判定で決まる。陽性率の差が効いているなら、差の大きい年齢でTPRとTNRが逆向きに開くはずである。

In [ ]:
def subgroup_performance(caches, split, masks):
    """seedごと・群ごとにAUROC・BACC・TPR・TNRと、スコアのずれを測る量を返す。

    implied_prevalenceは、class weightが加えるlog(w1/w0)を差し引いた確率の群平均で、
    モデルがその群の陽性率をいくつと見ているかの近似。重み付きCEで較正された場合にだけ正確で、
    較正の良さそのものは保証しない。

    Args:
        caches: (条件ID, seed)から予測cacheへの対応。
        split: cacheと同じ行順のtest split。
        masks: 群の所属。
    Returns:
        DataFrame: seed×群の指標。全体の行も含む。
    """
    target = split["target"].to_numpy()
    rows = []
    for (_, seed), cache in caches.items():
        probability = cache["probabilities"][:, 1]
        logit = cache["logits"][:, 1] - cache["logits"][:, 0]
        unweighted_probability = 1 / (1 + np.exp(-(logit - WEIGHT_LOG_ODDS)))
        members = {("overall", "overall"): np.ones(len(target), dtype=bool), **masks}
        for (attribute, subgroup), mask in members.items():
            row = performance(target[mask], probability[mask], cache["predictions"][mask])
            row.update(
                seed=seed,
                attribute=attribute,
                subgroup=subgroup,
                prevalence=target[mask].mean(),
                implied_prevalence=unweighted_probability[mask].mean(),
                median_logit_positive=np.median(logit[mask & (target == 1)]),
                median_logit_negative=np.median(logit[mask & (target == 0)]),
            )
            rows.append(row)
    return pd.DataFrame(rows)


by_seed = subgroup_performance(caches, frame, test_masks)
by_seed.to_csv(RESULTS / "subgroup_metrics_by_seed.csv", index=False)
METRIC_NAMES = {"auroc": "AUROC", "bacc": "BACC", "tpr": "TPR (sensitivity)", "tnr": "TNR (specificity)"}
stats = by_seed.groupby(["attribute", "subgroup"])[[*METRIC_NAMES, "implied_prevalence", "prevalence"]].agg(
    ["mean", "std"]
)
stats.to_csv(RESULTS / "subgroup_metrics_mean_std.csv")

fig, axes = plt.subplots(1, 4, figsize=(13, 3.4))
for ax, (metric, name) in zip(axes, METRIC_NAMES.items(), strict=True):
    for position, key in enumerate(SUBGROUPS):
        ax.errorbar(
            position,
            stats.loc[key, (metric, "mean")],
            yerr=stats.loc[key, (metric, "std")],
            fmt="o",
            color=ATTRIBUTE_COLORS[key[0]],
            capsize=3,
        )
    ax.axhline(stats.loc[("overall", "overall"), (metric, "mean")], color="gray", linestyle="--", linewidth=1)
    ax.axvline(1.5, color="lightgray", linewidth=0.8)
    ax.set_xticks(range(len(SUBGROUPS)), [subgroup for _, subgroup in SUBGROUPS])
    ax.set_title(name)
fig.suptitle(
    f"{MODEL_NAME}: subgroup performance (mean ± SD over 3 seeds; dashed = overall; left 2 = age, right 3 = race)"
)
fig.tight_layout()
fig.savefig(FIGURES / "subgroup_metrics.png", dpi=160)
plt.show()

年齢では、≥65のTPRが0.62と<65の0.89より大きく低く、TNRは逆に0.85と0.70で高い。BACCで見ると0.73と0.79の差になる。raceではTPR・TNRの差が3seedのばらつきと同程度で、はっきりしない。AUROCは≥65で低い（0.80と0.87）ので、年齢には順位の性能の差もある。raceではBlackのAUROCが0.84と低いが、Blackの陽性率は3群で最も高いので、陽性率では説明できない差である。

TPR・TNRの差が陽性率から来ているなら、群の陽性率を横軸に取ると、TPRは右上がり、TNRは右下がりに並ぶ（左図）。冒頭の2通りのどちらかは右図で区別する。モデルが見積もる群の陽性率（class weightが加えるlog(w1/w0)を差し引いた確率の平均）が、対角線に乗れば「反映している」、全体の見積もり（破線）に近い水平な並びなら「反映していない」。

In [ ]:
group_stats = stats.loc[SUBGROUPS]
fig, (left, right) = plt.subplots(1, 2, figsize=(11, 4))
for (attribute, subgroup), row in group_stats.iterrows():
    color = ATTRIBUTE_COLORS[attribute]
    x = row[("prevalence", "mean")]
    for metric, marker in [("tpr", "o"), ("tnr", "s")]:
        left.errorbar(x, row[(metric, "mean")], yerr=row[(metric, "std")], fmt=marker, color=color, capsize=3)
    left.annotate(subgroup, (x, row[("tpr", "mean")]), textcoords="offset points", xytext=(5, 5), fontsize=8)
    right.errorbar(
        x, row[("implied_prevalence", "mean")], yerr=row[("implied_prevalence", "std")], fmt="o", color=color, capsize=3
    )
    right.annotate(
        subgroup, (x, row[("implied_prevalence", "mean")]), textcoords="offset points", xytext=(5, -10), fontsize=8
    )
left.plot([], [], "o", color="gray", label="TPR")
left.plot([], [], "s", color="gray", label="TNR")
left.set_xlabel("subgroup positive rate (test)")
left.set_ylabel("rate at the shared threshold")
left.set_title("Positive rate vs TPR / TNR")
left.legend(fontsize=8)
limits = [0, group_stats[("prevalence", "mean")].max() * 1.2]
right.plot(limits, limits, color="gray", linewidth=1, label="tracks the subgroup rate (diagonal)")
overall_estimate = stats.loc[("overall", "overall"), ("implied_prevalence", "mean")]
right.axhline(
    overall_estimate, color="gray", linestyle="--", linewidth=1, label="shared across subgroups (overall estimate)"
)
right.set_xlabel("subgroup positive rate (test)")
right.set_ylabel("positive rate estimated by the model")
right.set_title("Mean probability after removing log(w1/w0)")
right.legend(fontsize=8)
fig.suptitle(f"{MODEL_NAME}: positive rate and operating point (blue = age, orange = race)")
fig.tight_layout()
fig.savefig(FIGURES / "prevalence_vs_operating_point.png", dpi=160)
plt.show()

右図では、5群とも対角線にほぼ乗った。≥65の見積もりは6.0%（実際5.7%）、<65は14.5%（実際14.4%）。ERMは画像から年齢を読み取り、**群ごとの陽性率をスコアに反映している**。少なくとも年齢については、「切片を全群で共有して較正がずれる」経路は当てはまらない。

すると≥65のTPRが低いのは、較正の合ったスコアを全群共通の閾値で切った結果になる。陽性率の低い群ではスコア全体が下がるので、同じ閾値では陽性を拾いにくい。

ただし、≥65はAUROCも低い。スコアの低下が陽性と陰性で一律（事前確率の分だけ）なのか、陽性側で余計に下がって分離が悪くなっているのかを、年齢の2群のlogitの分布で確かめる（raceは陽性率の差が小さいので省く）。一律なら、陽性・陰性の中央値はどちらも陽性率のlogitの差（約−1.03）だけ動くはずである。縦線はargmaxの閾値（logit差=0）。

In [ ]:
fig, axes = plt.subplots(1, len(SEEDS), figsize=(13, 3.4), sharey=True)
target = frame["target"].to_numpy()
for ax, seed in zip(axes, SEEDS, strict=True):
    cache = caches["M01", seed]
    logit = cache["logits"][:, 1] - cache["logits"][:, 0]
    for subgroup, color in [("<65", "C0"), ("≥65", "C3")]:
        mask = test_masks["age", subgroup]
        for label, style in [(1, "-"), (0, "--")]:
            values = np.sort(logit[mask & (target == label)])
            ax.plot(
                values,
                np.arange(1, len(values) + 1) / len(values),
                style,
                color=color,
                label=f"{subgroup}, {'positive' if label else 'negative'}",
            )
    ax.axvline(0, color="gray", linewidth=0.8)
    ax.set_title(f"seed {seed}")
    ax.set_xlabel("logit difference (class 1 − class 0)")
axes[0].set_ylabel("cumulative fraction")
axes[0].legend(fontsize=8)
fig.suptitle(f"{MODEL_NAME}: score distribution by age group (solid = positive, dashed = negative)")
fig.tight_layout()
fig.savefig(FIGURES / "age_logit_ecdf.png", dpi=160)
plt.show()

shift = by_seed[by_seed.attribute == "age"].pivot(
    index="seed", columns="subgroup", values=["median_logit_positive", "median_logit_negative"]
)
test_prevalence = prevalence.query("split == 'test' and attribute == 'age'").set_index("subgroup")["prevalence"]
prevalence_logit = np.log(test_prevalence / (1 - test_prevalence))
shift_summary = pd.DataFrame(
    {
        "陽性の中央値の差（≥65 − <65）": shift["median_logit_positive"]["≥65"] - shift["median_logit_positive"]["<65"],
        "陰性の中央値の差（≥65 − <65）": shift["median_logit_negative"]["≥65"] - shift["median_logit_negative"]["<65"],
        "陽性率のlogitの差（≥65 − <65）": prevalence_logit["≥65"] - prevalence_logit["<65"],
    }
)
shift_summary.to_csv(RESULTS / "age_logit_shift.csv")
print(shift_summary.round(3).to_string())

## まとめ

陽性の中央値は≥65で−1.07〜−1.52、陰性は−0.51〜−0.68動いた（seed 42〜44）。平均すると陽性率のlogitの差（−1.03）に近いが、陽性のほうが大きく下がっている。≥65のスコアの低下は、事前確率の分の一律のずれと、陽性と陰性の分離の悪化（AUROC 0.80）の両方を含む。

- 陽性率の差が大きいのは年齢（14.4%と5.7%）で、raceの差は小さい（9.5〜12.2%）。
- ERMは群の陽性率をスコアに反映しており、較正は群ごとにほぼ合っている。
- それでも全群共通の閾値では、陽性率の低い≥65でTPRが低く（0.62）、TNRが高い（0.85）。この差は較正のずれではなく、較正の合ったスコアを1本の閾値で切ることから生じる。≥65では分離の悪化も重なる。
- raceでは、TPR・TNRの差は3seedのばらつきと区別できない。BlackのAUROCの低さ（0.84）は陽性率と別の要因による。

公平性手法への含意は次の2つ。

- 較正が群ごとに合っていても、陽性率が違えば共通閾値でのTPR・TNRは揃わない（Kleinberg et al. 2016、Chouldechova 2017の両立不可能性）。TPRの差のうち陽性率と閾値で説明できる部分と、分離の悪さ（AUROC）による部分は分けて扱う。
- 群平均の損失は陽性率の高い群で大きくなる。DRO各条件のqは<65の4群に計83〜93%集まっており（[動作点の分析記録](reports/two_stage_operating_point.md)）、TPRの低い≥65ではなく陽性率の高い<65が「難しい群」として扱われている。

## TODO

- 群内のサンプリングの不確実性（bootstrap CI）を付ける。標準偏差は3seed間のばらつきだけを表す。
- 年齢×raceの交差群と、年齢の細かい区分で同じ関係が出るか。
- 群別閾値に変えたときにTPR・TNRの差がどこまで消えるか（切片のずれだけで説明できるかの直接の確認）。